In [ ]:
import numpy as np
import pandas as pd
from math import pi

from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.transform import dodge
from bokeh.palettes import Category20c
from bokeh.models import ColumnDataSource, HoverTool, Select
from bokeh.layouts import column, row
from bokeh.palettes import Category10
from bokeh.transform import factor_cmap
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool

output_notebook()

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

df = pd.read_csv('/kaggle/input/youmanitarian-international/volunteer_program_donations_dataset.csv')

# Convert necessary columns to numeric for plotting
df['HoursContributed'] = pd.to_numeric(df['HoursContributed'], errors='coerce')
df['TargetDonations'] = pd.to_numeric(df['TargetDonations'], errors='coerce')
df['TotalDonationsReceived'] = pd.to_numeric(df['TotalDonationsReceived'], errors='coerce')
df['AverageRating'] = pd.to_numeric(df['AverageRating'], errors='coerce')
df['DonationYear'] = pd.to_datetime(df['DateJoined'], dayfirst=True).dt.year

# Prepare a list of unique program names for filtering
programs = ['All'] + sorted(df['ProgramName'].unique().tolist())

# Initial data source (all data)
source = ColumnDataSource(df)

: 

In [ ]:
df = pd.read_csv('/kaggle/input/youmanitarian-international/volunteer_program_donations_dataset.csv')

# Display first 5 rows
print(df.head())

In [ ]:
# Ensure IsMember is in correct format
df['IsMember'] = df['IsMember'].astype(str)

member_counts = df['IsMember'].value_counts().reset_index()
member_counts.columns = ['IsMember', 'Count']

source_member = ColumnDataSource(member_counts)

p_member = figure(x_range=member_counts['IsMember'], height=300, title="Members vs Non-Members")
p_member.vbar(x='IsMember', top='Count', width=0.6, source=source_member, color="orange")

hover_member = HoverTool(tooltips=[("Status", "@IsMember"), ("Count", "@Count")])
p_member.add_tools(hover_member)

show(p_member)


In [ ]:
from math import pi

gender_counts = df['Gender'].value_counts().reset_index()
gender_counts.columns = ['Gender', 'Count']
gender_counts['angle'] = gender_counts['Count'] / gender_counts['Count'].sum() * 2 * pi
gender_counts['color'] = ['skyblue', 'lightpink']

p_gender = figure(height=400, title="Gender Distribution", toolbar_location=None, tools="hover", tooltips="@Gender: @Count", x_range=(-0.5, 1.0))
p_gender.wedge(x=0, y=1, radius=0.4, start_angle=cumsum('angle', include_zero=True),
               end_angle=cumsum('angle'), line_color="white", fill_color='color', legend_field='Gender',
               source=ColumnDataSource(gender_counts))
p_gender.axis.visible = False
p_gender.grid.grid_line_color = None
show(p_gender)


In [ ]:
role_counts = df['UserRole'].value_counts().reset_index()
role_counts.columns = ['UserRole', 'Count']

p_roles = figure(x_range=role_counts['UserRole'], height=300, title="Users per Role")
p_roles.vbar(x='UserRole', top='Count', width=0.5, source=ColumnDataSource(role_counts), color="mediumseagreen")
show(p_roles)


In [ ]:
# Group by ProgramName, sum HoursContributed
hours_per_program = df.groupby('ProgramName')['HoursContributed'].sum().reset_index()

source_bar = ColumnDataSource(hours_per_program)

p_bar = figure(x_range=hours_per_program['ProgramName'], height=350, title="Total Hours Contributed per Program",
               toolbar_location=None, tools="")

p_bar.vbar(x='ProgramName', top='HoursContributed', width=0.9, source=source_bar, fill_color='steelblue')

p_bar.xgrid.grid_line_color = None
p_bar.y_range.start = 0
p_bar.xaxis.major_label_orientation = 1.2
p_bar.yaxis.axis_label = "Hours Contributed"

hover_bar = HoverTool(tooltips=[("Program", "@ProgramName"), ("Hours", "@HoursContributed")])
p_bar.add_tools(hover_bar)

show(p_bar)


In [ ]:
df['DonationDate'] = pd.to_datetime(df['DateJoined'], dayfirst=True)
donation_trend = df.groupby(df['DonationDate'].dt.to_period('M'))['TotalDonationsReceived'].sum().reset_index()
donation_trend['DonationDate'] = donation_trend['DonationDate'].astype(str)

p_line = figure(x_range=donation_trend['DonationDate'], height=300, title="Monthly Donation Trend", x_axis_label='Month', y_axis_label='Total Donations')
p_line.line(x='DonationDate', y='TotalDonationsReceived', line_width=2, source=ColumnDataSource(donation_trend), color="firebrick")
p_line.xaxis.major_label_orientation = pi/4
show(p_line)


In [ ]:
avg_donations = df.groupby('UserRole')['TotalDonationsReceived'].mean().reset_index()
avg_donations.columns = ['UserRole', 'AvgDonation']

source = ColumnDataSource(avg_donations)

p_avg_donation = figure(y_range=avg_donations['UserRole'], height=300, title="Avg Donations per User Role",
                        x_axis_label="Average Donation", y_axis_label="User Role")

p_avg_donation.hbar(y='UserRole', right='AvgDonation', height=0.6, source=source, color="forestgreen")

show(p_avg_donation)

In [ ]:
program_counts = df['ProgramName'].value_counts().reset_index()
program_counts.columns = ['ProgramName', 'UserCount']

source = ColumnDataSource(program_counts)

p4 = figure(x_range=program_counts['ProgramName'], height=350, title="Count of users per Program")
p4.vbar(x='ProgramName', top='UserCount', width=0.6, source=source, color="mediumpurple")

p4.xaxis.major_label_orientation = 1.2
p4.xaxis.axis_label = "Program"
p4.yaxis.axis_label = "Number of Users"

show(p4)


In [ ]:
ratings = df.groupby('ProgramName')['AverageRating'].mean().reset_index()

source_rating = ColumnDataSource(ratings)

p_rating = figure(x_range=list(ratings['ProgramName']), height=350,
                  title="Average Rating per Program",
                  x_axis_label="Program", y_axis_label="Rating")

p_rating.line(x='ProgramName', y='AverageRating', source=source_rating, line_width=2, color="green")
p_rating.varea(x='ProgramName', y1=0, y2='AverageRating', source=source_rating, fill_color="green", fill_alpha=0.3)

hover_rating = HoverTool(tooltips=[("Program", "@ProgramName"), ("Avg Rating", "@AverageRating")])
p_rating.add_tools(hover_rating)

p_rating.xaxis.major_label_orientation = 1.2
p_rating.y_range.start = 0

show(p_rating)


In [ ]:
df['JoinMonth'] = pd.to_datetime(df['DateJoined'], dayfirst=True).dt.to_period('M').dt.to_timestamp()
monthly_new_users = df.groupby('JoinMonth').size().reset_index(name='NewUsers')

source = ColumnDataSource(monthly_new_users)

p = figure(x_axis_type='datetime', height=350, title="Monthly New Users",
           x_axis_label='Month', y_axis_label='Number of New Users')

p.line(x='JoinMonth', y='NewUsers', source=source, line_width=2, color='purple')
p.scatter(x='JoinMonth', y='NewUsers', source=source, size=6, color='purple')

show(p)

In [ ]:
df_sorted = df.sort_values('DateJoined')
df_sorted['JoinDate'] = pd.to_datetime(df_sorted['DateJoined'], dayfirst=True)

joined_over_time = df_sorted.groupby('JoinDate').size().cumsum().reset_index(name='CumulativeUsers')

source = ColumnDataSource(joined_over_time)

p = figure(x_axis_type='datetime', height=350, title="Users Joined Over Time (Cumulative)",
           x_axis_label='Date Joined', y_axis_label='Cumulative Users')

p.line(x='JoinDate', y='CumulativeUsers', source=source, line_width=2, color='teal')
p.scatter(x='JoinDate', y='CumulativeUsers', source=source, size=5, color='teal')

show(p)


In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource
import pandas as pd

location_counts = df['Location'].value_counts().reset_index()
location_counts.columns = ['Location', 'Count']

source = ColumnDataSource(location_counts)

p = figure(x_range=location_counts['Location'], title="User Count by Location", 
           x_axis_label='Location', y_axis_label='Number of Users', width=600, height=350)

p.vbar(x='Location', top='Count', width=0.7, source=source, color="skyblue")

p.xaxis.major_label_orientation = 1.2
p.xgrid.grid_line_color = None

show(p)


In [ ]:
# Step 1: Create SkillCount column
df['SkillCount'] = df['Skills'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)

# Step 2: Calculate histogram data
hist, edges = np.histogram(df['SkillCount'], bins=range(0, df['SkillCount'].max() + 2))

# Step 3: Prepare step plot data (x and y points for step lines)
x = []
y = []
for i in range(len(hist)):
    x.extend([edges[i], edges[i + 1]])
    y.extend([hist[i], hist[i]])

# Step 4: Create Bokeh plot
p = figure(title="Skill Count Distribution (Step Line)", height=350,
           x_axis_label='Number of Skills', y_axis_label='Number of Users')

p.line(x, y, line_width=3, color="navy")

show(p)

In [ ]:
statuses = df['IsMember'].dropna().unique().tolist()

stats = {}
for status in statuses:
    series = df[df['IsMember'] == status]['TotalDonationsReceived'].dropna()
    q1 = series.quantile(0.25)
    q2 = series.quantile(0.5)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    upper = min(q3 + 1.5 * iqr, series.max())
    lower = max(q1 - 1.5 * iqr, series.min())
    stats[status] = (q1, q2, q3, upper, lower)

q1s, q2s, q3s, uppers, lowers = zip(*stats.values())

source = ColumnDataSource(data=dict(
    status=statuses,
    q1=q1s,
    q2=q2s,
    q3=q3s,
    upper=uppers,
    lower=lowers,
))

p = figure(x_range=statuses, height=350, title="Donations Received by Membership Status",
           x_axis_label='Is Member', y_axis_label='Total Donations Received')

p.segment('status', 'upper', 'status', 'q3', source=source, line_color="black")
p.segment('status', 'lower', 'status', 'q1', source=source, line_color="black")

p.vbar('status', 0.7, 'q2', 'q3', source=source, fill_color="#E08E79", line_color="black")
p.vbar('status', 0.7, 'q1', 'q2', source=source, fill_color="#3B8686", line_color="black")

p.rect('status', 'lower', 0.2, 0.01, source=source, line_color="black")
p.rect('status', 'upper', 0.2, 0.01, source=source, line_color="black")

show(p)

In [ ]:
# Prepare data
df['JoinYear'] = pd.to_datetime(df['DateJoined'], dayfirst=True).dt.year
grouped = df.groupby(['JoinYear', 'Gender']).size().unstack(fill_value=0)

years = grouped.index.astype(str).tolist()
genders = grouped.columns.tolist()

source = ColumnDataSource(grouped)

p = figure(x_range=years, height=350, title="New Users by Gender per Year",
           x_axis_label='Year', y_axis_label='Number of New Users')

colors = ["#718dbf", "#e84d60", "#ddb7b1"]

for i, gender in enumerate(genders):
    p.vbar(x=dodge('x', -0.25 + i*0.25, range=p.x_range), top=gender, width=0.2, source=ColumnDataSource({
        'x': years,
        gender: grouped[gender].values
    }), color=colors[i], legend_label=gender)

p.x_range.range_padding = 0.1
p.xaxis.major_label_orientation = 1
p.legend.location = "top_left"
p.legend.title = "Gender"

show(p) 

In [ ]:
ages = df['Age'].dropna()
hist, edges = np.histogram(ages, bins=20)

p = figure(title="Age Distribution", height=350,
           x_axis_label='Age', y_axis_label='Number of Users')

p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], fill_color='skyblue', line_color='white', alpha=0.7)

show(p)


In [ ]:
# Group by Status and IsMember and count
grouped = df.groupby(['Status', 'IsMember']).size().unstack(fill_value=0)
grouped = grouped.reset_index()

source = ColumnDataSource(grouped)

status_list = grouped['Status'].tolist()
membership_list = ['Yes', 'No']
colors = ["#718dbf", "#e84d60"]

p = figure(x_range=status_list, height=350, title="User Count by Status and Membership",
           toolbar_location=None, tools="")

width = 0.4
for i, member in enumerate(membership_list):
    p.vbar(x=dodge('Status', -0.2 + i*width, range=p.x_range), top=member, width=width*0.9,
           source=source, color=colors[i], legend_label=member)

p.x_range.range_padding = 0.1
p.xgrid.grid_line_color = None
p.y_range.start = 0

p.xaxis.axis_label = "User Status"
p.yaxis.axis_label = "Number of Users"
p.legend.title = "Is Member"
p.legend.location = "top_right"

show(p)

In [ ]:
from collections import Counter

# Extract all skills
all_skills = df['Skills'].dropna().str.split(',').explode().str.strip()
top_skills = [skill for skill, _ in Counter(all_skills).most_common(5)]

# Prepare average rating per top skill
avg_ratings = []
for skill in top_skills:
    mask = df['Skills'].str.contains(skill, na=False)
    avg_rating = df.loc[mask, 'AverageRating'].mean()
    avg_ratings.append({'Skill': skill, 'AvgRating': avg_rating})

skill_df = pd.DataFrame(avg_ratings)

source = ColumnDataSource(skill_df)

p3 = figure(x_range=top_skills, title="Average Rating by Top Skills", 
            y_axis_label="Average Rating", width=500, height=350)

p3.vbar(x='Skill', top='AvgRating', width=0.6, source=source, color='orange')

p3.y_range.start = 0
p3.y_range.end = 5  # Assuming ratings out of 5
p3.xaxis.major_label_orientation = 1.2

show(p3)